In [1]:
from ngram_model import JelinekMercerModel
import logging 
from info_rate import count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config, create_minimal_summary
from syllabification import parse_to_phones_and_sylls
import numpy as np
import pickle
from pathlib import Path
from itertools import product
from joblib import Parallel, delayed
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline_jm(language, processing_type, text_type, n_values, folder_name, corpus_size):
    folder = Path(folder_name) / language
    config_dict = load_config(language) 
    #input_path = check_data_availability(language, processing_type, config_dict)

    (
        existing_ipa_path,
        phonized_path,
        syllabified_path,
        corpus_size_str,
        is_near_expected,
    ) = parse_to_phones_and_sylls(
        language=language,
        config_dict=config_dict,
        folder=folder,
        corpus_size=corpus_size,
    )

    if processing_type == 'words': 
        input_path = existing_ipa_path
    else: 
        input_path = phonized_path if processing_type == 'phones' else syllabified_path

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    if processing_type == 'words' and text_type == 'within_words':
        return 
    
    df_rows = []
    for n in n_values:
        jm_model = JelinekMercerModel(n)
        results = jm_model.fit_with_tuning(data, text_type, processing_type, language)

        logging.info(
        f"[n={n} | text_type={text_type} | processing_type={processing_type}] "
        f"Best λs (dev-tuned): {results['best_lambdas']} | "
        f"Mean IR: {np.mean(results['info_rate_values']):.3f} | "
        f"Dev PPL={results['dev_perplexity']:.3f} | "
        f"Test PPL={results['test_perplexity']:.3f}"
        )

        # Display a table with the computed numbers
        metrics = {
        "ID": results["info_density"],
        "IR": np.mean(results["info_rate_values"]),
        'test_perplexity': results["test_perplexity"],
        "dev_perplexity": results["dev_perplexity"],
        }

        # Add each lambda_k as its own numeric metric
        for k, v in results["best_lambdas"].items():
            metrics[f"lambda_{k}"] = v

        # Append to df_rows in long format (Metric, Value)
        df_rows.extend([
            {
                "Language": language,
                "UnitType": processing_type,
                "TextType": text_type,
                "n": n,
                "Metric": metric,
                "Value": round(value, 6) if isinstance(value, (int, float)) else value
            }
            for metric, value in metrics.items()
        ])

    return pd.DataFrame(df_rows) if df_rows else None 

In [2]:
# Create all combinations to process, adjust as needed
languages = ['ENG']
processing_types = ['sylls', 'phones']
text_types = ['within_words', 'across_sentences']
n_values = [1,2,3]  
folder_name = "produced_data_large_corpus" 
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=1, backend="loky", verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline_jm)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)
print("\n" + "═" * 90)
print("📊 Results for Jelinek–Mercer Smoothed n-Gram Model   ".center(90, " "))
print("    λ values tuned via grid search on development set         ".center(90, " "))
print("═" * 90 + "\n")

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(
    r is not None or (proc == "words" and txt == "within_words")
    for (lang, proc, txt), r in zip(tasks, results)
)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

✅ Found largest IPA corpus for ENG: produced_data_large_corpus/ENG/ipa_corpus_ENG_size:450000.pkl
⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 450000.


INFO:root:[Split] within_words | total=450000 sentences → train=432000, dev=9000, test=9000
INFO:root:[Tune for train] λ_1 = 1.0 | dev PPL=618.274
INFO:root:Best λs for n=1: {1: 1.0} (dev ppl=618.274)
INFO:root:Training vocabulary size: 11344
INFO:root:[Final Eval] Best λs: {1: 1.0} |Test PPL=616.998 | within_words
INFO:root:[n=1 | text_type=within_words | processing_type=sylls] Best λs (dev-tuned): {1: 1.0} | Mean IR: 53.839 | Dev PPL=618.274 | Test PPL=616.998
INFO:root:[Split] within_words | total=450000 sentences → train=432000, dev=9000, test=9000
INFO:root:[Tune for train] λ_1 = 1.0 | dev PPL=1306.437
INFO:root:[Tune for train] λ_2 = 0.9 | dev PPL=11.702
INFO:root:Best λs for n=2: {1: 1.0, 2: 0.9} (dev ppl=11.702)
INFO:root:Training vocabulary size: 11344
INFO:root:[Final Eval] Best λs: {1: 1.0, 2: 0.9} |Test PPL=11.787 | within_words
INFO:root:[n=2 | text_type=within_words | processing_type=sylls] Best λs (dev-tuned): {1: 1.0, 2: 0.9} | Mean IR: 25.052 | Dev PPL=11.702 | Test PP

✅ Found largest IPA corpus for ENG: produced_data_large_corpus/ENG/ipa_corpus_ENG_size:450000.pkl
⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 450000.


INFO:root:[Split] across_sentences | total=450000 sentences → train=432000, dev=9000, test=9000
INFO:root:[Tune for train] λ_1 = 1.0 | dev PPL=618.274
INFO:root:Best λs for n=1: {1: 1.0} (dev ppl=618.274)
INFO:root:Training vocabulary size: 11344
INFO:root:[Final Eval] Best λs: {1: 1.0} |Test PPL=616.998 | across_sentences
INFO:root:[n=1 | text_type=across_sentences | processing_type=sylls] Best λs (dev-tuned): {1: 1.0} | Mean IR: 53.839 | Dev PPL=618.274 | Test PPL=616.998
INFO:root:[Split] across_sentences | total=450000 sentences → train=432000, dev=9000, test=9000
INFO:root:[Tune for train] λ_1 = 1.0 | dev PPL=691.208
INFO:root:[Tune for train] λ_2 = 0.9 | dev PPL=73.373
INFO:root:Best λs for n=2: {1: 1.0, 2: 0.9} (dev ppl=73.373)
INFO:root:Training vocabulary size: 11344
INFO:root:[Final Eval] Best λs: {1: 1.0, 2: 0.9} |Test PPL=73.012 | across_sentences
INFO:root:[n=2 | text_type=across_sentences | processing_type=sylls] Best λs (dev-tuned): {1: 1.0, 2: 0.9} | Mean IR: 37.260 | D

AttributeError: 'Parallel' object has no attribute '_pre_dispatch_amount'

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG']
processing_types = ['words']
text_types = ['across_sentences']
n_values = [1,2,3,4]  
folder_name = "produced_data_large_corpus" 
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=1, backend="loky", verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline_jm)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)
print("\n" + "═" * 90)
print("📊 Results for Jelinek–Mercer Smoothed n-Gram Model   ".center(90, " "))
print("    λ values tuned via grid search on development set         ".center(90, " "))
print("═" * 90 + "\n")

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(
    r is not None or (proc == "words" and txt == "within_words")
    for (lang, proc, txt), r in zip(tasks, results)
)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

: 

: 

: 

In [ ]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config, create_minimal_summary
from syllabification import parse_to_phones_and_sylls
import numpy as np
import pickle
from pathlib import Path
import logging
from itertools import product
from joblib import Parallel, delayed
import pandas as pd
import psutil
import os

# PARALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def print_memory_usage(label=""):
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / 1024 / 1024  # in MB
    print(f"[{label}] Memory usage: {mem:.2f} MB")


def run_pipeline(language, processing_type, text_type, n_values, folder_name, corpus_size):
    print_memory_usage(f"Start {language}-{processing_type}-{text_type}")
    folder = Path(folder_name) / language
    config_dict = load_config(language) 
   
    (
        existing_ipa_path,
        phonized_path,
        syllabified_path,
        corpus_size_str,
        is_near_expected,
    ) = parse_to_phones_and_sylls(
        language=language,
        config_dict=config_dict,
        folder=folder,
        corpus_size=corpus_size,
    )

    #input_path = check_data_availability(language, processing_type, config_dict)

    if processing_type == 'words': 
        input_path = existing_ipa_path
    else: 
        input_path = phonized_path if processing_type == 'phones' else syllabified_path

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    if processing_type == 'words' and text_type == 'within_words':
        return 

    markov_models = {}
    df_rows = []
    lower_models_cache = {}

    for n in n_values:

        # Create and build the Markov model
        model = MarkovModel(n)

        # Store lower-level models as cache
        if n > 1 and (n - 1) not in lower_models_cache:
            lower_model = MarkovModel(n - 1)
            lower_model.build(data, text_type)
            lower_models_cache[n - 1] = lower_model

            # Build the markov model
            model.build(data, text_type, lower_model=lower_models_cache.get(n - 1))

        # Compute the conditional entropy (information density)
        info_density = model.compute_conditional_entropy()
        #logger.info(f"Information Density: {info_density:.4f}")
        print_memory_usage(f"ID Comp {language}-{processing_type}-{text_type}")

        # Compute the information rate (bits per second)
        info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
        #logger.info(f"🧮 {language} | corpus size: {corpus_size_str} | {processing_type} | {text_type} | n={n} | IR: {np.mean(info_rate_values):.4f}")
        
        # Display a table with the computed numbers
        metrics = {
            "ID": info_density,
            "IR": np.mean(info_rate_values),
        }

        df_rows.extend([
            {
                "Language": language,
                "UnitType": processing_type,
                "TextType": text_type,
                "n": n,
                "Metric": metric,
                "Value": round(value, 4),
            }
            for metric, value in metrics.items()
        ])

        # Save the results for a corpus with largest possible size 
        """if is_near_expected: 
            update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
            update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
            update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type) 
            print_memory_usage(f"Save to csv {language}-{processing_type}-{text_type}")

            # Store model for later use 
            markov_models[n] = model
            
            # Save the model to a file
            model.save_model(language, folder, processing_type, text_type, corpus_size_str)"""
    
    print_memory_usage(f"End {language}-{processing_type}-{text_type}")
    return pd.DataFrame(df_rows) if df_rows else None      


In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG']
processing_types = ['sylls']
text_types = ['within_words', 'across_sentences']
n_values = [2,3]  
folder_name = "produced_data_large_corpus" 
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=15, backend="loky", verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

print(f"-------- Results for bucketed Jelinek-Mercer smoothed n-gram model with λs trained via EM --------")
# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG', 'FRA', 'DEU']
processing_types = ['phones', 'sylls']
text_types = ['within_words', 'across_sentences']
n_values = [1,2,3,4]  
folder_name = "produced_data" 
corpus_size =  5000 # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

In [ ]:
from helpers import clean_corpus_size_files
clean_corpus_size_files("produced_data", ["ENG", "FRA", "DEU"], 2000, ['phones', 'sylls'])

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG', 'FRA', 'DEU']
processing_types = ['words']
text_types = ['across_sentences']
n_values = [1,2]  
folder_name = "produced_data_large_corpus" 
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=15, backend="loky", verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

print(f"-------- Results for alpha = 0.000001 --------")
# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['DEU']
processing_types = ['words']
text_types = ['across_sentences']
n_values = [1,2]  
folder_name = "produced_data_large_corpus"  
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=15, verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")